# Reconciliación del conteo de reseñas duplicadas: 413 vs 418

**Objetivo:** en `1_analisis_exploratorio_y_entrenamiento.ipynb` se reportaron **418 filas repetidas** (texto 100% idéntico). En `revies_duplicadas.ipynb` se reportaron **413** usando un criterio distinto (coincidencia de las primeras 150 palabras). Este notebook reconcilia ambas cifras, identifica de dónde viene la diferencia, incorpora un argumento de **probabilidad** sobre qué tan factible es que palabras coincidan por azar en la misma secuencia, y concluye cuántas reseñas están **realmente** duplicadas.

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from math import comb
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
%matplotlib inline

df = pd.read_csv('data/IMDB Dataset.csv')
df['id'] = df.index
print(f"Dataset original: {df.shape[0]:,} filas")

Dataset original: 50,000 filas


## 1. Igualar criterios antes de comparar

Antes de comparar 413 contra 418 hay que separar **dos cosas que se mezclaron**:

1. **La métrica reportada:** ¿es un conteo de *grupos* de reseñas repetidas, o de *filas extra* (copias más allá de la primera aparición)? No son lo mismo si algún grupo tiene 3, 4 o 5 copias.
2. **El criterio de duplicidad:** ¿coincidencia del **texto completo** (100% idéntico) o solo de las **primeras 150 palabras** (un criterio más laxo, que puede confundir reseñas distintas que solo comparten el inicio)?

Recalculamos ambos criterios con ambas métricas para poder comparar de forma consistente.

In [2]:
def prefijo(texto, k):
    return ' '.join(str(texto).split()[:k])

df['p20'] = df['review'].apply(lambda x: prefijo(x, 20))
df['p150'] = df['review'].apply(lambda x: prefijo(x, 150))

def resumen_duplicados(df, columna_clave, sentiment_col='sentiment'):
    g = df.groupby([columna_clave, sentiment_col])['id'].count()
    grupos = g[g > 1]
    n_grupos = len(grupos)
    n_filas_totales = grupos.sum()
    n_filas_extra = n_filas_totales - n_grupos
    return n_grupos, n_filas_totales, n_filas_extra, grupos

n_grupos_full, n_filas_full, n_extra_full, grupos_full = resumen_duplicados(df, 'review')
n_grupos_150, n_filas_150, n_extra_150, grupos_150 = resumen_duplicados(df, 'p150')
n_grupos_20, n_filas_20, n_extra_20, grupos_20 = resumen_duplicados(df, 'p20')

tabla = pd.DataFrame({
    'Criterio': ['Texto 100% idéntico', 'Primeras 150 palabras', 'Primeras 20 palabras'],
    'Grupos': [n_grupos_full, n_grupos_150, n_grupos_20],
    'Filas totales en grupos': [n_filas_full, n_filas_150, n_filas_20],
    'Filas "extra" (a eliminar)': [n_extra_full, n_extra_150, n_extra_20],
})
tabla

,Criterio,Grupos,Filas totales en grupos,"Filas ""extra"" (a eliminar)"
0,Texto 100% idéntico,406,824,418
1,Primeras 150 palabras,413,839,426
2,Primeras 20 palabras,443,902,459


**Ya aparecen las dos cifras en juego:** con el criterio de texto 100% idéntico hay **406 grupos** y **418 filas extra**. Con el criterio de 150 palabras hay **413 grupos**. Es decir:

- El "418" del notebook 1 = **filas extra** bajo el criterio **exacto**.
- El "413" de `revies_duplicadas.ipynb` = **grupos** bajo el criterio de **150 palabras**.

Se compararon dos cosas distintas en dos dimensiones a la vez. Vamos a separar cada dimensión.

## 2. Dimensión 1 — Por qué "grupos" ≠ "filas extra" (mismo criterio)

Un grupo de tamaño $n$ (n copias idénticas) aporta **1 grupo** pero **$n-1$ filas extra**. Si todos los grupos fueran pares ($n=2$), grupos y filas extra coincidirían. La diferencia surge de los grupos con 3 o más copias.

In [3]:
dist_tam_full = grupos_full.value_counts().sort_index()
dist_tam_full.index.name = 'Tamaño del grupo (copias idénticas)'
dist_tam_full = dist_tam_full.rename('Número de grupos')
print("Distribución de tamaños de grupo — criterio EXACTO:")
print(dist_tam_full)

tamanos = dist_tam_full.index.to_numpy()
contrib_extra = ((tamanos - 1) * dist_tam_full.values).sum()
print(f"\nVerificación: 406 grupos, pero filas extra = sum((tamaño-1) x n_grupos) = {contrib_extra}")
print(f"406 grupos de tamaño 2 aportarían 397 filas extra; los grupos de 3, 4 y 5 copias aportan las {418-397} filas extra restantes.")

Distribución de tamaños de grupo — criterio EXACTO:
Tamaño del grupo (copias idénticas)
2    397
3      7
4      1
5      1
Name: Número de grupos, dtype: int64

Verificación: 406 grupos, pero filas extra = sum((tamaño-1) x n_grupos) = 418
406 grupos de tamaño 2 aportarían 397 filas extra; los grupos de 3, 4 y 5 copias aportan las 21 filas extra restantes.


Con esto queda explicada la primera parte de la brecha: **406 grupos exactos generan 418 filas extra**, no porque haya un error, sino porque 9 de esos grupos tienen 3, 4 o incluso 5 copias en vez de 2. *(406 grupos de tamaño 2 solamente contarían 397; los grupos más grandes agregan las 21 filas extra restantes hasta llegar a 418.)*

## 3. Dimensión 2 — Por qué el criterio de 150 palabras encuentra 413 grupos y no 406

Comparando ahora **grupos contra grupos** (misma métrica): 406 (exacto) vs 413 (150 palabras) → una diferencia de **7 grupos**. El criterio de 150 palabras es más laxo: agrupa reseñas que **comparten el inicio** pero pueden **divergir después**. Buscamos explícitamente esos casos.

In [4]:
casos_parciales = []
for key, g in df.groupby(['p150', 'sentiment']):
    if len(g) > 1 and g['review'].nunique() > 1:
        casos_parciales.append((key, g[['id', 'review']].values.tolist()))

print(f"Grupos (criterio 150 palabras) donde el texto completo NO es idéntico entre todos sus miembros: {len(casos_parciales)}")
for key, miembros in casos_parciales:
    ids = [m[0] for m in miembros]
    print(f"  IDs {ids}  (sentiment={key[1]})")

Grupos (criterio 150 palabras) donde el texto completo NO es idéntico entre todos sus miembros: 8
  IDs [13499, 23900]  (sentiment=negative)
  IDs [1328, 2811]  (sentiment=positive)
  IDs [21720, 22635]  (sentiment=positive)
  IDs [28097, 29865]  (sentiment=negative)
  IDs [3216, 14214]  (sentiment=negative)
  IDs [7446, 19242, 34847]  (sentiment=negative)
  IDs [14540, 26342]  (sentiment=positive)
  IDs [20545, 42400]  (sentiment=positive)


Encontramos **8** grupos "mixtos" (no 7): en 7 de ellos, ninguna de las reseñas es realmente idéntica a otra (falsos positivos puros); en 1 de ellos, dos de las tres reseñas SÍ son idénticas entre sí (ya contabilizado dentro de los 406 exactos) y la tercera solo comparte el inicio. La aritmética cierra así:

$$ \underbrace{405}_{\text{grupos 150-palabras 100\% idénticos}} + \underbrace{1}_{\text{grupo mixto con un par exacto adentro}} = \underbrace{406}_{\text{grupos exactos totales}} $$
$$ \underbrace{405}_{\text{100\% idénticos}} + \underbrace{7}_{\text{falsos positivos puros}} + \underbrace{1}_{\text{mixto}} = \underbrace{413}_{\text{grupos por 150 palabras}} $$

Veamos 2 ejemplos concretos de los "falsos positivos": reseñas que comparten el inicio pero **no son la misma reseña**.

In [5]:
def mostrar_par(ids, max_car=220):
    for i in ids:
        texto = df.loc[i, 'review']
        print(f"ID {i} ({len(texto)} caracteres):")
        print(f"  INICIO: {texto[:max_car]}...")
        print(f"  FINAL : ...{texto[-max_car:]}")
        print()

print("=== Caso A: mismo inicio, desarrollo distinto ===")
mostrar_par([28097, 29865])

print("=== Caso B: mismo inicio, uno con nota/calificación añadida ===")
mostrar_par([14540, 26342])

=== Caso A: mismo inicio, desarrollo distinto ===
ID 28097 (1641 caracteres):
  INICIO: In a college dorm a guy is killed by somebody with a scythe. His girlfriend Beth (Dorie Barton) discovers him and tries to commit suicide. She's institutionalized. A year later she's out, has a new boyfriend named Hank (...
  FINAL : ...talk about sex; Baston's non reaction to seeing a friend getting killed is kind of funny and WHAT happens to Lawrence? His character disappears without a trace at the end! Dull, stupid, no gore, no nudity--skip this one.

ID 29865 (1701 caracteres):
  INICIO: In a college dorm a guy is killed by somebody with a scythe. His girlfriend Beth (Dorie Barton) discovers him and tries to commit suicide. She's institutionalized. A year later she's out, has a new boyfriend named Hank (...
  FINAL : ...ting killed is kind of funny and WHAT happens to Lawrence? His character disappears without a trace at the end! Dull, stupid, no gore, no nudity--skip this one.<br /><br />Rated 

Se confirma: estas reseñas **no son duplicados reales**. Comparten un inicio (probablemente porque un mismo usuario reutilizó/actualizó su reseña, o por una frase de apertura habitual), pero el contenido completo difiere — divergen en el desarrollo o tienen texto añadido al final. Contarlas como "duplicadas" sería un error de limpieza: se estaría borrando contenido único.

## 4. Argumento de probabilidad: ¿pueden coincidir 150 (o 20) palabras por puro azar?

Aquí entra el punto que pediste: ¿qué tan probable es que se repita una **secuencia** de palabras por casualidad, sin que exista una relación real entre los textos?

**Modelo simplificado:** si cada palabra de una reseña se sorteara de forma independiente según la frecuencia con la que aparece en el corpus (esto es una simplificación — el lenguaje real no es así, tiene gramática y frases fijas — pero sirve como cota de referencia), la probabilidad de que **dos reseñas coincidan en una palabra específica de una posición** es:
$$ P(\text{colisión por palabra}) = \sum_{w} p_w^2 $$
(el índice de colisión de Simpson: para cada palabra $w$ del vocabulario con probabilidad empírica $p_w$, la probabilidad de que ambas reseñas "sorteen" esa misma palabra en esa posición es $p_w^2$; se suma sobre todo el vocabulario).

Si asumimos independencia entre posiciones (otra simplificación, ver nota después), la probabilidad de que **coincidan las $k$ palabras seguidas** es aproximadamente:
$$ P(\text{coincidencia de una secuencia de } k \text{ palabras}) \approx \left(\sum_w p_w^2\right)^k $$

In [6]:
muestra = df['review'].sample(n=8000, random_state=42)
contador = Counter()
total_tokens = 0
for texto in muestra:
    palabras = str(texto).lower().split()
    contador.update(palabras)
    total_tokens += len(palabras)

probs = np.array(list(contador.values())) / total_tokens
colision = (probs ** 2).sum()

print(f"Vocabulario estimado (muestra de 8,000 reseñas): {len(contador):,} palabras distintas")
print(f"P(colisión por palabra, bajo modelo i.i.d.) ~ {colision:.6f}  (aprox. 1 en {1/colision:.0f})")

n_reviews = df.shape[0]
n_pares_posibles = comb(n_reviews, 2)

for k in [20, 150]:
    p_k = colision ** k
    esperado = n_pares_posibles * p_k
    print(f"\nk={k} palabras seguidas:")
    print(f"  P(coincidencia exacta por puro azar) ~ {p_k:.3e}")
    print(f"  Pares de reseñas posibles en el dataset: {n_pares_posibles:,.0f}")
    print(f"  Número esperado de coincidencias por AZAR en todo el dataset ~ {esperado:.3e}")

Vocabulario estimado (muestra de 8,000 reseñas): 125,121 palabras distintas
P(colisión por palabra, bajo modelo i.i.d.) ~ 0.007796  (aprox. 1 en 128)

k=20 palabras seguidas:
  P(coincidencia exacta por puro azar) ~ 6.869e-43
  Pares de reseñas posibles en el dataset: 1,249,975,000
  Número esperado de coincidencias por AZAR en todo el dataset ~ 8.586e-34

k=150 palabras seguidas:
  P(coincidencia exacta por puro azar) ~ 5.980e-317
  Pares de reseñas posibles en el dataset: 1,249,975,000
  Número esperado de coincidencias por AZAR en todo el dataset ~ 7.475e-308


**Lectura del resultado:** incluso para una secuencia de solo **20 palabras**, el número de coincidencias que esperaríamos ver *por puro azar* en las ~1.25 mil millones de parejas de reseñas del dataset es del orden de $10^{-34}$ — es decir, **cero, para cualquier propósito práctico**. Para 150 palabras, la probabilidad es aún más ínfima (del orden de $10^{-317}$).

**Conclusión estadística:** ninguna de las coincidencias que encontramos (443 grupos con 20 palabras, 413 con 150, 406 con texto completo) puede explicarse por azar. Toda coincidencia larga de texto refleja un **origen común real**: copia exacta, republicación de la misma reseña, o reutilización de una plantilla/inicio.

**Pero "origen común" no es lo mismo que "es la misma reseña".** El modelo i.i.d. usado arriba es además una **cota inferior** de la probabilidad real de coincidencia accidental, porque el lenguaje natural no es i.i.d.: existen frases de apertura convencionales ("I watched this movie and...", "This film is about...") que dos personas distintas pueden usar **por convención del idioma**, no por copiar. Eso explica, con números reales de este dataset, por qué entre más corta la secuencia exigida, se encuentran más grupos:

- 20 palabras → 443 grupos (más frases de apertura genéricas coinciden)
- 150 palabras → 413 grupos (menos coincidencias, pero aún incluye 7-8 falsos positivos, ya identificados arriba)
- Texto completo → 406 grupos (el único criterio que garantiza duplicado real, sin falsos positivos)

## 5. Conclusión final

**¿Cuántas reseñas están realmente duplicadas?**

> **406 reseñas** tienen al menos una copia con **texto 100% idéntico** (mismo `review` y mismo `sentiment`). Esas 406 reseñas están involucradas en **824 filas** del dataset, de las cuales **418 son copias "extra"** que deberían eliminarse si se quiere una fila única por reseña.

**Por qué el "413" y el "418" no se contradicen, solo miden cosas distintas:**

| # | Fuente | Qué mide | Valor |
|---|---|---|---|
| 1 | Notebook 1 | Filas extra, texto 100% idéntico | **418** |
| 2 | `reviews_duplicadas.ipynb` | Grupos, prefijo de 150 palabras (criterio más laxo) | **413** |
| — | Este notebook | Grupos, texto 100% idéntico (criterio correcto para "duplicado real") | **406** |
| — | Este notebook | Filas extra, prefijo de 150 palabras | 426 |

- La diferencia **406 → 418** (dentro del mismo criterio exacto) se debe a que 9 grupos tienen 3, 4 o 5 copias en vez de 2 (una fila extra por cada copia adicional más allá de la primera).
- La diferencia **406 → 413** (misma métrica de "grupos", pero cambiando de criterio) se debe a que el criterio de 150 palabras incluye **7 falsos positivos** verificados: reseñas que comparten el inicio pero **no son la misma reseña completa** (contenido añadido o desarrollo distinto), confirmado con probabilidad prácticamente nula de que eso ocurra por azar en el sentido de coincidencia *aleatoria sin relación* — pero sabemos, por inspección directa, que su relación es "comparten una plantilla/inicio", no "son la misma reseña".

**Recomendación práctica:** para limpiar el dataset antes de vectorizar (como se hizo en el notebook 1), usar el **criterio de texto 100% idéntico** (`drop_duplicates` sobre `review` + `sentiment`): elimina exactamente **418 filas**, dejando **49,582 filas únicas**. El criterio de prefijo de palabras es útil como **señal exploratoria** de reutilización de contenido (para auditoría manual), pero **no debe usarse para eliminar filas automáticamente**, porque borraría reseñas con contenido único.